In [2]:
import pandas as pd
from pathlib import Path

pasta = Path(".")
arquivos = sorted(
    arquivo for arquivo in pasta.glob("*.csv")
    if arquivo.name != "todos.csv"
)

dados = pd.concat(
    [pd.read_csv(arquivo, sep=";") for arquivo in arquivos],
    ignore_index=True
)

dados.to_csv("todos.csv", index=False, sep=";")
dados.head()

KeyboardInterrupt: 

Vou localizar o arquivo `2021.csv` e verificar o conteúdo inicial e a codificação, porque esse tipo de erro ao abrir CSV no VS Code costuma vir de separador, encoding ou caminho do arquivo. Depois faço a correção mais pequena e testo a abertura.

Investigar erro do CSV

Read memory

A busca automática foi cancelada antes de retornar dados; vou fazer uma checagem direta e objetiva no arquivo indicado para confirmar a causa.

Read [](file:///c%3A/Users/Robson%20Am%C3%A9rico/Documents/SCTEC/Mini%20Projeto%202)

Read [](file:///c%3A/Users/Robson%20Am%C3%A9rico/Documents/GitHub/Projeto_Final)

Read [](file:///c%3A/Users/Robson%20Am%C3%A9rico/Documents/GitHub/Futebol)

Read [](file:///c%3A/Users/Robson%20Am%C3%A9rico/Documents/SCTEC/Mini%20Projeto%202/Readme.md)

Read [](file:///c%3A/Users/Robson%20Am%C3%A9rico/Documents/SCTEC/Mini%20Projeto%202/2021.csv#1-1), lines 1 to 12

Read [](file:///c%3A/Users/Robson%20Am%C3%A9rico/Documents/SCTEC/Mini%20Projeto%202/2020.csv#1-1), lines 1 to 4

O arquivo não está vazio nem claramente corrompido: ele usa `;` como separador e mistura campos com e sem aspas, algo comum em exportações brasileiras. Vou agora confirmar tamanho, encoding e se alguma linha tem número de colunas diferente; isso separa um problema de configuração do Data Wrangler de uma corrupção real do CSV.

Ran terminal command: $file = 'c:\Users\Robson Américo\Documents\SCTEC\Mini Projeto 2\2021.csv'; Get-Item $file | Select-Object FullName,Length; $bytes = [System.IO.File]::ReadAllBytes($file); [BitConverter]::ToString($bytes[0..([Math]::Min(15,$bytes.Length-1))]); $reader = [System.IO.StreamReader]::new($file,$true); $null = $reader.ReadLine(); $reader.CurrentEncoding.WebName; $reader.Close()

Ran terminal command: $file = 'c:\Users\Robson Américo\Documents\SCTEC\Mini Projeto 2\2021.csv'; $lines = Get-Content -LiteralPath $file; $counts = $lines | ForEach-Object { ($_ -split ';').Count } | Group-Object | Sort-Object Name; $counts | Select-Object Name,Count

Ran terminal command: Get-ChildItem -LiteralPath 'c:\Users\Robson Américo\Documents\SCTEC\Mini Projeto 2' -Filter '*.csv' | Select-Object Name,Length | Sort-Object Name

O `2021.csv` parece válido, mas tem cerca de **52,8 MB** e usa `;` como separador. O erro do Data Wrangler provavelmente ocorre pelo tamanho ou pela detecção automática do separador.

Abra no notebook com:



In [ ]:
import pandas as pd

df_2021 = pd.read_csv(
    "2021.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

df_2021.head()



No Excel/Power BI, importe como:

- Delimitador: `;`
- Codificação: `UTF-8`
- Primeira linha como cabeçalho

Se o Data Wrangler continuar falhando, use uma amostra menor:



In [ ]:
df_amostra = pd.read_csv("2021.csv", sep=";", encoding="utf-8", nrows=10000)



O arquivo não aparenta estar corrompido; o ponto principal é o tamanho e o delimitador `;`.

In [ ]:
import pandas as pd

arquivo_final = pd.read_csv("todos.csv", sep=";")
nulos_por_coluna = arquivo_final.isna().sum()
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0].sort_values(ascending=False)
quantidade_duplicatas = arquivo_final.duplicated().sum()

print(f"Linhas: {len(arquivo_final)}")
print(f"Colunas: {len(arquivo_final.columns)}")
print(f"Duplicatas completas: {quantidade_duplicatas}")

print("\nNulos por coluna:")
if colunas_com_nulos.empty:
    print("Nenhum valor nulo encontrado.")
else:
    display(colunas_com_nulos.to_frame(name="quantidade_nulos"))

Linhas: 367570
Colunas: 36
Duplicatas completas: 0

Nulos por coluna:


,quantidade_nulos
nu_ata,278418
vl_capacidade,233803
sg_unidade_medida,233803
registro_anvisa,179718
fg_generico,179718
ds_observacao,58710
dt_insercao,2142
co_classe,398
co_pdm,398
no_classe,398


In [ ]:
colunas_texto = [
    "nu_ata",
    "sg_unidade_medida",
    "registro_anvisa",
    "fg_generico",
    "ds_observacao",
    "no_instituicao",
    "nu_processo_compra"
]

for coluna in colunas_texto:
    arquivo_final[coluna] = arquivo_final[coluna].fillna("Não informado")

arquivo_final["ds_observacao"] = arquivo_final["ds_observacao"].replace(
    "Não informado", "Sem observação"
)

arquivo_final["sg_unidade_medida"] = arquivo_final["sg_unidade_medida"].fillna(
    "Não se aplica"
)

arquivo_final.to_csv("todos_tratado.csv", index=False, sep=";")